<a href="https://colab.research.google.com/github/NadaMagdy-hub/Olist-E-Commerce-Data-Analysis/blob/main/geolocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import unicodedata
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
geolocation = pd.read_csv('olist_geolocation_dataset.csv')

In [ ]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [ ]:
geolocation.tail()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS
1000162,99950,-28.070104,-52.018658,tapejara,RS


In [ ]:
geolocation.columns.tolist()

['geolocation_zip_code_prefix',
 'geolocation_lat',
 'geolocation_lng',
 'geolocation_city',
 'geolocation_state']

In [ ]:
geolocation.shape

(1000163, 5)

In [ ]:
geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


In [ ]:
geolocation.duplicated().sum()



np.int64(261831)

In [ ]:
geolocation.drop_duplicates(inplace=True)

In [ ]:
geolocation.duplicated().sum()

np.int64(0)

------معرفة اذ كانت الخليه فارغه ام لا ----------

In [ ]:
geolocation.isna().sum()

,0
geolocation_zip_code_prefix,0
geolocation_lat,0
geolocation_lng,0
geolocation_city,0
geolocation_state,0


In [ ]:
geolocation.describe()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
count,738332.000000,738332.000000,738332.000000
mean,38316.086800,-20.998353,-46.461098
std,30632.496675,5.892315,4.393705
min,1001.000000,-36.605374,-101.466766
25%,12600.000000,-23.603061,-48.867822
50%,29144.000000,-22.873588,-46.647278
75%,65950.000000,-19.923336,-43.836974
max,99990.000000,45.065933,121.105394


------فلترة الإحداثيات لتكون داخل حدود البرازيل فقط-------

In [ ]:
brazil_lat_limit = (-33.75, 5.27)
brazil_lng_limit = (-73.98, -34.79)
geolocation = geolocation[
    geolocation['geolocation_lat'].between(brazil_lat_limit[0], brazil_lat_limit[1]) &
    geolocation['geolocation_lng'].between(brazil_lng_limit[0], brazil_lng_limit[1])
]

In [ ]:
geolocation.shape

(738299, 5)

------لتوحيد وتطهير أسماء المدن (إزالة الهمزات والمسافات)-----

In [ ]:
def clean_city_name(text):
    if pd.isna(text):
        return text
    text = str(text).lower().strip()
    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )
    return text
geolocation['geolocation_city'] = geolocation['geolocation_city'].apply(clean_city_name)

----إذا كان لدينا نفس الموقع مكرراً في عدة أسطر بإحداثيات مختلفة قليلاً، فسيقوم الكود بحساب المتوسط الحسابي (mean) لخطوط الطول والعرض للحصول على نقطة متوسطة تمثل هذا المكان بدقة ثم يقوم بدمج وتطهير بياناتك جغرافياً (Geographic Aggregation) بالاعتماد على الرمز البريدي (geolocation_zip_code_prefix)-----

In [ ]:
agg_rules = {
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0],
    'geolocation_state': lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]
}
geolocation_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(agg_rules).reset_index()


In [ ]:
geolocation.describe(include='object')

,geolocation_city,geolocation_state
count,738299,738299
unique,5964,27
top,sao paulo,SP
freq,99645,285975


In [ ]:
geolocation_clean.to_csv('olist_geolocation_cleaned.csv', index=False)
print(geolocation_clean.head())

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1001       -23.550227       -46.634039   
1                         1002       -23.547657       -46.634991   
2                         1003       -23.549000       -46.635582   
3                         1004       -23.549829       -46.634792   
4                         1005       -23.549547       -46.636406   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  


In [ ]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
